In [54]:
import os
import streamlit as st
import pickle
import time
import langchain
from langchain import OpenAI
from langchain.chains import RetrievalQAWithSourcesChain
from langchain.chains.qa_with_sources.loading import load_qa_with_sources_chain
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import UnstructuredURLLoader
from langchain.embeddings import OpenAIEmbeddings
from langchain.vectorstores import FAISS

In [ ]:
#load openAI api key
os.environ['OPENAI_API_KEY'] = 'Your-OpenAI-API-Key'

In [58]:
pip install -U langchain-openai

Note: you may need to restart the kernel to use updated packages.


In [60]:
from langchain_openai import OpenAI
llm = OpenAI(temperature=0.9, max_tokens=500)

In [62]:
loaders = UnstructuredURLLoader(urls=[
    "https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html",
    "https://www.moneycontrol.com/news/business/tata-motors-launches-punch-icng-price-starts-at-rs-7-1-lakh-11098751.html"
])
data = loaders.load() 
len(data)

2

In [63]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=1000,
    chunk_overlap=200
)

# As data is of type documents we can directly use split_documents over split_text in order to get the chunks.
docs = text_splitter.split_documents(data)
len(docs)

16

In [64]:
docs[0]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/markets/wall-street-rises-as-tesla-soars-on-ai-optimism-11351111.html'}, page_content='English\n\nHindi\n\nGujarati\n\nSpecials\n\nHello, Login\n\nHello, Login\n\nLog-inor Sign-Up\n\nMy Account\n\nMy Profile\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nMy Profile\n\nMy PRO\n\nMy Portfolio\n\nMy Watchlist\n\nMy Alerts\n\nMy Messages\n\nPrice Alerts\n\nLogout\n\nLoans up to ₹50 LAKHS\n\nFixed Deposits\n\nCredit CardsLifetime Free\n\nCredit Score\n\nChat with Us\n\nDownload App\n\nFollow us on:\n\nNetwork 18\n\nGo Ad-Free\n\nMy Alerts\n\n>->MC_ENG_DESKTOP/MC_ENG_NEWS/MC_ENG_MARKETS_AS/MC_ENG_ROS_NWS_MKTS_AS_ATF_728\n\nMoneycontrol\n\nGo PRO NowPRO\n\nMoneycontrol PRO\n\nAdvertisement\n\nRemove Ad\n\nBusiness\n\nMarkets\n\nStocks\n\nEconomy\n\nCompanies\n\nTrends\n\nIPO\n\nOpinion\n\nEV Special\n\nLoans\n\nLoans\n\nHomeNewsBusinessMarketsWall Street rises as Tesla soars on AI optimism

In [68]:
from langchain_openai import OpenAIEmbeddings

# Create the embeddings of the chunks using openAIEmbeddings
embeddings = OpenAIEmbeddings()

# Pass the documents and embeddings inorder to create FAISS vector index
vectorindex_openai = FAISS.from_documents(docs, embeddings)

In [104]:
# Save FAISS index safely
vectorindex_openai.save_local("faiss_store_openai")

In [106]:
vectorindex_openai = FAISS.from_documents(docs, embeddings)
vectorindex_openai.save_local("faiss_store_openai")  # ✅ ADD THIS

In [110]:
from langchain.vectorstores import FAISS
from langchain.embeddings import OpenAIEmbeddings

embeddings = OpenAIEmbeddings()

# ✅ Load the vector index safely
vectorindex_openai = FAISS.load_local(
    "faiss_store_openai",
    embeddings,
    allow_dangerous_deserialization=True  # ✅ trust your own data
)

In [114]:
retriever = vectorindex_openai.as_retriever()
docs = retriever.get_relevant_documents("what is the price of Tiago iCNG?")

In [116]:
for i, doc in enumerate(docs):
    print(f"\nDocument {i+1}:\n")
    print(doc.page_content)


Document 1:

The company also said it has also introduced the twin-cylinder technology on its Tiago and Tigor models.

The Tiago iCNG is priced between Rs 6.55 lakh and Rs 8.1 lakh, while the Tigor iCNG comes at a price range of Rs 7.8 lakh to Rs 8.95 lakh.

Tata Motors Passenger Vehicles Ltd Head-Marketing, Vinay Pant said these introductions put together will make the company's CNG line up "appealing, holistic, and stronger than ever".

PTI

first published: Aug 4, 2023 02:17 pm

Discover the latest Business News, Budget 2025 News, Sensex, and Nifty updates. Obtain Personal Finance insights, tax queries, and expert opinions on Moneycontrol or download the Moneycontrol App to stay updated!

Advertisement

Remove Ad

Advertisement

Remove Ad

Advertisement

Remove Ad

Advertisement

Remove Ad

Advertisement

Remove Ad

Advertisement

Remove Ad

Advisory Alert:

moneycontrol

Follow Us On:

Facebook

twitter

instagram

linkedin

telegram

youtube

Document 2:

Set Alert

live

bselive